# **11762 Content-Based Image Retrieval**
## Master's Degree in Intelligent Systems
### University of the Balearic Islands
---

##### Write in the following the names of the members of the group:
- **Member 1:** Einar López Altamirano
- **Member 2:** Victor Armando Canales Lima

# **Instructions**

You can find a **pre-configured Conda environment on Aula Digital** with everything required to complete the assignments.

**Do not delete any of the provided cells or functions**. Write your code in the indicated sections. You may add new cells or functions as needed to complete your work.  

Along with this assignment, please submit a comprehensive **report** in PDF format explaining your implemented solutions. The report should include, for example:

- **Technical Explanation**: Clearly describe the algorithms and techniques used, including the relevant code snippets.
- **Results**: Present your results in a clear and concise manner. Use visualizations such as graphs or tables to enhance understanding.
- **Analysis**: Interpret your results and discuss their significance. Consider, for instance, the following questions:
  - Are the results as expected? Why or why not?
  - What factors might have influenced them?
- **Conclusions**: Summarize your findings and provide overall conclusions about your project.
- ---

In [2]:
# Execute this cell to make sure 
# that external modules are reloaded
%load_ext autoreload
%autoreload 2

In [3]:
# Fill the following variable with
# the path to the Holidays dataset
dataset_dir = '/data/image-indexing-11762/holidays_mini'

In [4]:
# Setup code for this assignment
import cv2
import math
import numpy as np
import os
import scipy.cluster.vq as vq
import zipfile
from holidays_dataset_handler import HolidaysDatasetHandler

import assignment2 as a2

# Configuring Matplotlib
from matplotlib import pyplot as plt
%matplotlib inline

# **Introduction**
In this assignment, you will implement and evaluate different methods for image retrieval using the **INRIA Holidays dataset**. Refer to **Assignment 1** for additional details on the dataset structure and the class `HolidaysDatasetHandler`, that will be used again in this assignment for managing the dataset and to evaluate the performance of various indexing methods.

## Loading Keypoints and Descriptors
Unlike in **Assignment 1**, you will now be provided with a set of **precomputed SIFT descriptors** for each image. Therefore, you do not need to extract these features from scratch as we did previously. Instead, you will use the provided **SIFT keypoints and descriptors** along with the dataset.

To utilize these precomputed features, simply load them when initializing a `HolidaysDatasetHandler` object by setting the `load_features` parameter to `True`, as shown below:

In [5]:
# Load dataset with precomputed SIFT features
dataset = HolidaysDatasetHandler(dataset_dir, load_features=True)

Loading features ...
Completed!


This will ensure that all necessary keypoints and descriptors are available for retrieval tasks without requiring additional computation.

> **Some images do not have keypoints / descriptors. Take this into account when you develop your solution.**

# **$k$-d Trees**

In this section, you will use a set of randomized $k$-d trees to index a database of images. 

- To achieve this, start by implementing the function `build_kdtree` in the `assignment2.py` file. This function will take a dataset as input and construct a set of randomized $k$-d trees, using the FLANN library format.

> **Useful links**: [cv2.FlannBasedMatcher](https://docs.opencv.org/4.5.5/dc/de2/classcv_1_1FlannBasedMatcher.html), [Possible algorithms to create an index](https://docs.opencv.org/4.5.5/db/d18/classcv_1_1flann_1_1GenericIndex.html#a8fff14185f9f3d2f2311b528f65b146c), [Algorithm IDs](https://github.com/opencv/opencv/blob/master/modules/flann/include/opencv2/flann/defines.h#L70)

- Next, implement the function `search_kdtree` in the `assignment2.py` file. This function performs a nearest neighbor search using FLANN (previously trained with `build_kdtree`) on the provided query descriptors, and applies Lowe's ratio test to filter weak matches. It returns a list of image indices, ordered by the number of matches found, to identify the most relevant images in the database.

**Task 1:**

Using the `HolidaysDatasetHandler` and the functions you have implemented, your task in the following cell is to compute the **mean Average Precision (mAP)** of the image retrieval system for this dataset, utilizing an FLANN-based kd-tree index. To reduce the computational load of the notebook, limit the number of descriptors per image to **400**. Use **4** trees.

In [ ]:
# Fill this variable with the resulting mAP
mAP_kdtree = 0.0

# YOUR CODE HERE
raise NotImplementedError()
# -----

print('Mean Average Precision (mAP): %.5f' % mAP_kdtree)

**Task 2:**

Are the results consistent? Do you get the same mAP every time? Explain why this may or may not happen when running the procedure multiple times using the same trees, without rebuilding them.

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

**Task 3:**

Analyze the impact of varying the number of trees on both mAP and average response time. Visualizing the results with plots could help support your analysis.

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

**Task 4:**

Analyze the effect of changing the number of descriptors per image on both mAP and average response time. Keep in mind that using too many descriptors may consume **significant memory** on your computer. Visualizing the results with plots could help justify your analysis.

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

# **Locality-Sensitive Hashing (LSH)**

In this section, we will explore how to use **Locality-Sensitive Hashing (LSH)** to efficiently index and search an image database. We will utilize **[FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search)**, an optimized library for large-scale similarity search, which is widely used for tasks such as image retrieval, nearest neighbor search, and clustering.

LSH is a technique that maps high-dimensional vectors into a lower-dimensional space using **random projections**. The key idea behind LSH is to ensure that similar vectors (in terms of Euclidean or Cosine distance) are **hashed into the same or nearby buckets**, allowing for faster retrieval. FAISS implements LSH by **projecting each descriptor onto a set of random hyperplanes**. The sign of each projection (positive or negative) determines a **binary code** for that descriptor, effectively mapping it into a lower-dimensional Hamming space.

One of the most important parameters in LSH indexing in FAISS is **`nbits`**, which defines the **final number of bits** each descriptor is converted into and, hence, the **number of hyperplanes** used for hashing.

- Start by implementing the function `build_lsh` in the `assignment2.py` file, which will create an LSH index using FAISS and store the image descriptors efficiently for fast retrieval.

> **Useful links**: [FAISS Tutorial](https://github.com/facebookresearch/faiss/wiki/getting-started), [Random Projection for Locality Sensitive Hashing](https://www.pinecone.io/learn/series/faiss/locality-sensitive-hashing-random-projection/)

- Next, implement the function `search_lsh` in the `assignment2.py` file. Unlike previous implementations, this function searches the LSH index for the best matching images for multiple query images simultaneously, as FAISS is more efficient when processing batch queries instead of one at a time.

**Task 5:**

Using the `HolidaysDatasetHandler` and the functions you have implemented, your task in the following cell is to compute the **mean Average Precision (mAP)** of the image retrieval system for this dataset, utilizing an FAISS-based LSH index. For this exercise, we will extract **400 descriptors per image** for efficiency and start with an initial configuration of **`nbits = 128`**, meaning each descriptor will be mapped into a **128-bit hash**.

In [ ]:
# Fill this variable with the resulting mAP
mAP_lsh = 0.0

# YOUR CODE HERE
raise NotImplementedError()
# -----

print('Mean Average Precision (mAP) using LSH:', round(mAP_lsh, 5))

**Task 6:**

Analyze the impact of changing the number of bits (i.e., the number of projections and hash size) on both **mAP (Mean Average Precision)** and **average response** time. Since using an excessively high number of bits can significantly increase memory consumption, discretize a few key values and study their effects. To support your analysis, consider visualizing the results with plots, as this can help illustrate trends and justify your conclusions.

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

**Task 7:**

Compare the performance of **randomized k-d trees** and **LSH** from multiple perspectives, including accuracy, training time, and querying time. Since each approach has different trade-offs, analyzing their efficiency across these aspects will provide valuable insights. Visualizing the results with plots can be particularly useful to support and justify your conclusions.

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

# **Bag-of-Words (BoW)**

In this exercise, you will implement an image retrieval system based on the **Bag-of-Words (BoW) model** using SIFT descriptors extracted from images. The system will include:

- A **visual vocabulary** loaded from a precomputed `.fvecs` file.
- A **FAISS L2 index** to efficiently quantize descriptors and assign them to visual words.
- An **inverted file index** to enable fast lookup of relevant images.
- **TF-IDF weighting** to enhance retrieval accuracy by prioritizing discriminative visual words.

### Visual Vocabularies

To implement a **Bag-of-Words (BoW) model**, we first need a **visual vocabulary**. The authors of the **INRIA Holidays dataset** provide precomputed visual vocabularies, trained using a clustering method (e.g., **$k$-means**) on a separate dataset (**Flickr60K**). These vocabularies are included in the dataset directory under the **`vocabs/`** subfolder. Each file corresponds to a vocabulary of different sizes, containing 100, 200, 500, 1K, 2K, 5K, 10K, 20K, 50K, 100K, and 200K visual words. Since these vocabularies are stored in binary format, we will provide functions to correctly load them into your program.

###  Implementing the Model

Start by implementing the missing methods in the `BagOfWordsRetriever` class in `assignment2.py` as indicated.

**Task 8:**

Using the `HolidaysDatasetHandler` and `BagOfWordsRetriever` classes, your task in the following cell is to compute the **mean Average Precision (mAP)** of the image retrieval system for this dataset **using the vocabularies of 200, 2K, 10K and 20K visual words**. Depending on your machine, this may take some time, as we are performing a linear search over the dictionary. Please allow some time for this part of the notebook to execute. You may try a different FAISS index, but please note that it could affect the mAP.

In [ ]:
# Fill these variables with the resulting mAP
mAP_200  = 0.0
mAP_2K   = 0.0
mAP_10K  = 0.0
mAP_20K  = 0.0

# YOUR CODE HERE
raise NotImplementedError()
# -----

print('mAP 200: %.5f' % mAP_200)
print('mAP 2K: %.5f' % mAP_2K)
print('mAP 10K: %.5f' % mAP_10K)
print('mAP 20K: %.5f' % mAP_20K)

**Task 9:**

Compare the performance results obtained for each case. Is a larger vocabulary size always advantageous? Justify your answer.

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

**Task 10:**

Analyze the impact of vocabulary size on mAP and average response time (including both training and query times). Are these times consistent across different vocabulary sizes? Including plots here could help support your answer.

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

**Task 11:**

Do the results depend on the set of images used to generate the vocabulary? How can retrieval performance be improved?

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

**Task 12:**

How does TF-IDF impact performance? Does it improve or degrade results? Does this outcome make sense?

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

# **Deep Learning Retrieval with HNSW**

In this exercise, you will implement a modern image retrieval system based on **Deep Features** extracted using a Convolutional Neural Network (CNN) and indexed with a Hierarchical Navigable Small World (**HNSW**) graph. The system will include:

- Deep Feature Extraction: Using a pre-trained **ResNet-50** model to obtain high-dimensional semantic vectors (embeddings) for each image.

- HNSW Indexing: A graph-based index implemented with **FAISS** to enable fast approximate nearest neighbor (ANN) search in high-dimensional spaces.

### Deep Features
Unlike local descriptors (e.g., SIFT) which require quantization into a visual vocabulary, Deep Learning models map entire images directly into a compact vector space where semantic similarity is preserved. We will use again the ResNet-50 architecture, pre-trained on ImageNet, to extract a 2048-dimensional embedding for every image in the dataset. Since extracting these features can be computationally expensive in this assignment, this time you are provided with a precomputed file (`holidays_deep_features.npy`) containing the feature vectors for the entire dataset.

To load these features, use the following code:
```python
deep_features = np.load('holidays_deep_features.npy', allow_pickle=True).item()
```

The raw vectors from the CNN are not normalized. Since we want to use Cosine Similarity as our similarity measure, you must apply L2 normalization to both the database vectors and the query vectors before indexing or searching.

- To achieve this, start by implementing the function `build_hnsw` in the `assignment2.py` file. This function will take the dataset handler and the dictionary of deep features as input. It must extract the database vectors, perform L2 normalization, and construct an HNSW index using FAISS (faiss.IndexHNSWFlat). You must configure the index with the number of connections $M=32$ and use the faiss.METRIC_INNER_PRODUCT to simulate Cosine Similarity. The function should return the built index and the image map (the list of image names corresponding to the indexed vectors).

> **Useful links**: [FAISS HNSW Index](https://github.com/facebookresearch/faiss/wiki/The-index-factory)

- Next, implement the function `search_hnsw` in the `assignment2.py` file. This function performs an approximate nearest neighbor search using the HNSW index (previously trained with `build_hnsw`) on the provided query descriptors (passed as the first argument). It returns the distances and indices of the $k$ nearest neighbors. Ensure you apply the same L2 normalization to the query vectors before searching.

**Task 13:**

Using the `HolidaysDatasetHandler` and the functions you have implemented, your task in the following cell is to compute the **mean Average Precision (mAP)** of the image retrieval system for this dataset using the HNSW index and the Deep descriptors.

In [ ]:
# Fill these variables with the resulting mAP
mAP_hnsw  = 0.0

# YOUR CODE HERE
raise NotImplementedError()
# -----

print('Mean Average Precision (mAP) using HNSW: %.5f' % mAP_hnsw)

**Task 14:**

Analyze the effect of HNSW parameters on performance. Evaluate the approach in terms of speed and computational cost. How does it compare to previous solutions?

*Write in the following the code required to answer the question. You may add more code or markdown cells as needed.*

## Submitting Your Work

**Important**: Please ensure that the notebook has been run and that the **cell outputs are visible**.

**Important**: Additionally, make sure you have filled in the names at the beginning of the notebook and the **ID** variable in the following cell.

Once you have completed the necessary code and are satisfied with your solution, **save your notebook** and run the following cell:

In [ ]:
ID = '99999999R' # Your DNI or NIE

zip_filename = ID + '_A2.zip'
zf = zipfile.ZipFile(zip_filename, mode = 'w')

zf.write('11762_Image_Indexing.ipynb');
zf.write('assignment2.py');
zf.write('holidays_dataset_handler.py');
zf.write('holidays_deep_features.npy');

zf.close()

This will generate a zip file of your code called `ID_A2.zip` in the same directory of the assignment. This is the file that you must upload to [Aula Digital](https://ad.uib.es/) to submit your work!

---

&copy; Emilio Garcia-Fidalgo, University of the Balearic Islands